
# One-Rec — embed shard 0/5

Pure embedding worker for the catalog-expansion queue: fresh preview URL per
track (they expire in ~15 min), Discogs-EffNet embeddings via a **packed
batch-64** runner (equivalence-gated against the reference implementation,
with automatic fallback), npz checkpoints. Takes every 5th queue row
starting at index 60,000.


In [ ]:
import asyncio, glob, json, os, time
from pathlib import Path

import numpy as np
import pandas as pd

%pip install -q essentia-tensorflow==2.1b6.dev1389 aiohttp
!wget -q -nc https://essentia.upf.edu/models/feature-extractors/discogs-effnet/discogs-effnet-bs64-1.pb
print("model:", os.path.getsize("discogs-effnet-bs64-1.pb") / 1e6, "MB")
T_SESSION = time.time()

CFG = dict(
    shard=0,
    n_shards=5,
    shard_start=60000,
    deezer_rps=8,
    batch=512,            # download/embed unit, inside the preview-URL TTL
    ckpt_every=2_000,
    mel_workers=4,
    time_budget_h=11.2,
    seed=42,
)
WORK = Path("/kaggle/working")
time.sleep(CFG["shard"] * 7)  # stagger shard start-up against shared rate limits


In [ ]:
def input_glob(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

queue = pd.read_parquet(input_glob("embed_queue.parquet")[0])

done_ids = set()
for f in input_glob("ext_ckpt_*.npz") + sorted(str(p) for p in WORK.glob("ext_ckpt_*.npz")):
    z = np.load(f, allow_pickle=True)
    done_ids.update(z["ids"].tolist())
miss_ids = set()
for p in input_glob("ext_misses*.parquet") + sorted(str(x) for x in WORK.glob("ext_misses*.parquet")):
    miss_ids |= set(pd.read_parquet(p)["id"])

tail = queue.iloc[CFG["shard_start"]:].reset_index(drop=True)
mine = tail[np.arange(len(tail)) % CFG["n_shards"] == CFG["shard"]]
todo = mine[~mine["id"].isin(done_ids) & ~mine["id"].isin(miss_ids)].reset_index(drop=True)
print(f"queue {len(queue):,} | shard rows {len(mine):,} | already done {len(done_ids):,} "
      f"| known misses {len(miss_ids):,} | todo {len(todo):,}")


## Deezer client — fresh URL per track; only permanent misses are recorded

In [ ]:
import aiohttp

HDRS = {"User-Agent": "one-rec-research/1.0"}


class RateLimiter:
    def __init__(self, rps):
        self.min_int = 1.0 / rps
        self.next_t = 0.0
        self.lock = asyncio.Lock()

    async def acquire(self):
        async with self.lock:
            now = time.monotonic()
            wait = self.next_t - now
            self.next_t = max(now, self.next_t) + self.min_int
        if wait > 0:
            await asyncio.sleep(wait)


DEEZER = RateLimiter(CFG["deezer_rps"])


async def deezer_get(session, url, params=None, retries=6):
    for attempt in range(retries):
        await DEEZER.acquire()
        try:
            async with session.get(url, params=params, timeout=aiohttp.ClientTimeout(total=15)) as resp:
                data = await resp.json(content_type=None)
        except Exception:
            await asyncio.sleep(2 * (attempt + 1))
            continue
        if isinstance(data, dict) and data.get("error", {}).get("code") == 4:
            await asyncio.sleep(5 + 3 * attempt)  # shared-IP quota pressure: back off harder
            continue
        return data
    return None


DL_SEM = asyncio.Semaphore(16)


async def fetch_blob(session, dzid):
    """(blob, permanent_miss). Rate-limit/timeouts are TRANSIENT — never
    recorded as misses, the track just stays queued for the next pass."""
    d = await deezer_get(session, f"https://api.deezer.com/track/{dzid}")
    if d is None:
        return None, False                      # transient: API unreachable/quota
    if not isinstance(d, dict) or not d.get("preview"):
        return None, True                       # permanent: no preview on Deezer
    async with DL_SEM:
        for _ in range(3):
            try:
                async with session.get(d["preview"], timeout=aiohttp.ClientTimeout(total=30)) as resp:
                    if resp.status == 200:
                        return await resp.read(), False
            except Exception:
                pass
            await asyncio.sleep(1)
    return None, False                          # transient: CDN hiccup


async def _download_batch(rows):
    async with aiohttp.ClientSession(headers=HDRS) as session:
        out = await asyncio.gather(*(fetch_blob(session, d) for d in rows["deezer_id"]))
    return [(i, blob, perm) for i, (blob, perm) in zip(rows["id"], out)]


def download_batch(rows):
    return asyncio.run(_download_batch(rows))


## Embedders — reference (proven, ~1 clip/s) and packed batch-64 (~4x)

The packed path is only used if it reproduces the reference embeddings on real
clips (cosine > 0.999) — the space is frozen, a mismatch would poison the
catalog silently.

In [ ]:
import multiprocessing as mp
import tempfile

# ---------------- reference path: fork pool, one padded forward per clip ----
_model = None


def _init_worker():
    global _model, _MonoLoader
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
    os.environ["TF_NUM_INTEROP_THREADS"] = "1"
    from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs
    _MonoLoader = MonoLoader
    _model = TensorflowPredictEffnetDiscogs(graphFilename="discogs-effnet-bs64-1.pb",
                                            output="PartitionedCall:1")


def _decode(blob):
    f = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    try:
        f.write(blob)
        f.close()
        from essentia.standard import MonoLoader
        return MonoLoader(filename=f.name, sampleRate=16000, resampleQuality=4)()
    finally:
        os.unlink(f.name)


def _embed_one(args):
    tid, blob = args
    try:
        audio = _decode(blob)
        if len(audio) < 16000:
            return tid, None
        return tid, _model(audio).mean(axis=0).astype(np.float32)
    except Exception:
        return tid, None


# ---------------- packed path: mel in fork pool, packed TF in main ----------
_mel = None


def _init_mel():
    global _mel
    os.environ["OMP_NUM_THREADS"] = "1"
    from essentia.standard import TensorflowInputMusiCNN
    _mel = TensorflowInputMusiCNN()


def _mel_one(args):
    tid, blob = args
    try:
        audio = _decode(blob)
        if len(audio) < 16000:
            return tid, None
        from essentia.standard import FrameGenerator
        bands = np.array([_mel(f) for f in FrameGenerator(audio, frameSize=512, hopSize=256,
                                                          startFromZero=True)], np.float32)
        return tid, bands
    except Exception:
        return tid, None


class PackedEffnet:
    """Full batch-64 forwards over patches packed across clips."""

    def __init__(self, patch_hop, tail_mode):
        from essentia import Pool
        from essentia.standard import TensorflowPredict
        self.Pool = Pool
        self.predict = TensorflowPredict(graphFilename="discogs-effnet-bs64-1.pb",
                                         inputs=["serving_default_melspectrogram"],
                                         outputs=["PartitionedCall:1"])
        self.hop, self.tail = patch_hop, tail_mode

    def patches(self, bands):
        P = 128
        n_full = 1 + (len(bands) - P) // self.hop if len(bands) >= P else 0
        out = [bands[i * self.hop:i * self.hop + P] for i in range(n_full)]
        used = (n_full - 1) * self.hop + P if n_full else 0
        if self.tail != "discard" and used < len(bands):
            rest = bands[n_full * self.hop:] if n_full else bands
            if len(rest) >= 16:  # skip sub-patch slivers shorter than ~0.25s
                if self.tail == "zeros":
                    pad = np.zeros((P - len(rest), bands.shape[1]), np.float32)
                elif self.tail == "edge":
                    pad = np.repeat(rest[-1:], P - len(rest), axis=0)
                else:  # cycle
                    reps = int(np.ceil(P / len(rest)))
                    rest = np.tile(rest, (reps, 1))[:P]
                    pad = np.zeros((0, bands.shape[1]), np.float32)
                out.append(np.concatenate([rest, pad]) if len(pad) else rest)
        return out

    def _forward(self, batch64):
        pool = self.Pool()
        pool.set("melspectrogram", batch64[:, None, :, :])  # (64, 1, 128, 96)
        out = self.predict(pool)["PartitionedCall:1"]
        return np.asarray(out).reshape(64, -1)

    def embed_many(self, items):
        """items: [(tid, bands)] -> [(tid, mean 1280-d vec)]"""
        owners, patch_list = [], []
        for tid, bands in items:
            for p in self.patches(bands):
                owners.append(tid)
                patch_list.append(p)
        sums, counts = {}, {}
        for lo in range(0, len(patch_list), 64):
            chunk = patch_list[lo:lo + 64]
            batch = np.zeros((64, 128, 96), np.float32)
            batch[:len(chunk)] = np.stack(chunk)
            embs = self._forward(batch)[:len(chunk)]
            for tid, e in zip(owners[lo:lo + 64], embs):
                sums[tid] = sums.get(tid, 0.0) + e
                counts[tid] = counts.get(tid, 0) + 1
        return [(tid, (sums[tid] / counts[tid]).astype(np.float32)) for tid in dict.fromkeys(owners)]


## Equivalence gate — derive patch hop + tail mode from the reference on
real clips; require cosine > 0.999 or fall back

In [ ]:
os.environ["TF_NUM_INTRAOP_THREADS"] = "4"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

ref_pool = mp.get_context("fork").Pool(CFG["mel_workers"], initializer=_init_worker)
mel_pool = mp.get_context("fork").Pool(CFG["mel_workers"], initializer=_init_mel)

PACKED = None
if len(todo):
    sample = download_batch(todo.head(8))
    sample_ok = [(tid, blob) for tid, blob, _ in sample if blob][:4]
    if len(sample_ok) >= 3:
        ref_out = {tid: emb for tid, emb in ref_pool.map(_embed_one, sample_ok) if emb is not None}
        mel_out = {tid: bands for tid, bands in mel_pool.map(_mel_one, sample_ok) if bands is not None}

        # reference patch counts per clip pin down (hop, tail) exactly
        from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs
        ref_model = TensorflowPredictEffnetDiscogs(graphFilename="discogs-effnet-bs64-1.pb",
                                                   output="PartitionedCall:1")
        ref_counts = {}
        for tid, blob in sample_ok:
            if tid in mel_out:
                ref_counts[tid] = len(ref_model(_decode(blob)))

        best = None
        for hop in (128, 96, 64, 62, 32):
            for tail in ("discard", "zeros", "edge", "cycle"):
                probe = PackedEffnet(hop, tail)
                if any(len(probe.patches(mel_out[t])) != ref_counts[t] for t in ref_counts):
                    continue
                packed_out = dict(probe.embed_many([(t, mel_out[t]) for t in ref_counts]))
                cos = [float(packed_out[t] @ ref_out[t]
                             / (np.linalg.norm(packed_out[t]) * np.linalg.norm(ref_out[t])))
                       for t in ref_counts if t in ref_out]
                print(f"hop={hop} tail={tail}: counts match, mean-vec cos={min(cos):.5f}")
                if min(cos) > 0.999:
                    best = (hop, tail)
                    break
            if best:
                break
        if best:
            PACKED = PackedEffnet(*best)
            print(f"PACKED path ON (hop={best[0]}, tail={best[1]})")
        else:
            print("PACKED equivalence FAILED — falling back to the proven per-clip path")
    else:
        print("not enough sample clips — using the proven per-clip path")

if PACKED is None:
    mel_pool.close()


def embed_batch(payloads):
    if PACKED is not None:
        bands = [(tid, b) for tid, b in mel_pool.map(_mel_one, payloads) if b is not None]
        return PACKED.embed_many(bands)
    return [(tid, emb) for tid, emb in ref_pool.map(_embed_one, payloads) if emb is not None]


## Embed loop — per-shard checkpoints, time-budgeted

In [ ]:
from concurrent.futures import ThreadPoolExecutor

SHARD_TAG = f"s{CFG['shard']}"
pending_ids, pending_embs, new_misses = [], [], []
emb_total = 0


def flush(force=False):
    global pending_ids, pending_embs
    if pending_ids and (force or len(pending_ids) >= CFG["ckpt_every"]):
        n = len(list(WORK.glob(f"ext_ckpt_{SHARD_TAG}_*.npz")))
        np.savez(WORK / f"ext_ckpt_{SHARD_TAG}_{n:03d}.npz",
                 ids=np.array(pending_ids, dtype=object),
                 embs=np.stack(pending_embs).astype(np.float16))
        pending_ids, pending_embs = [], []
    if new_misses:
        pd.DataFrame({"id": sorted(set(new_misses))}).to_parquet(
            WORK / f"ext_misses_{SHARD_TAG}.parquet", index=False)


batches = [todo.iloc[i:i + CFG["batch"]] for i in range(0, len(todo), CFG["batch"])]
throughput_printed = False
with ThreadPoolExecutor(max_workers=1) as fetcher:
    future = fetcher.submit(download_batch, batches[0]) if batches else None
    for bi in range(len(batches)):
        triples = future.result()
        if bi + 1 < len(batches):
            future = fetcher.submit(download_batch, batches[bi + 1])
        payloads = [(i, b) for i, b, _ in triples if b]
        new_misses.extend(i for i, b, perm in triples if not b and perm)
        t0 = time.time()
        embedded = embed_batch(payloads)
        for tid, emb in embedded:
            pending_ids.append(tid)
            pending_embs.append(emb)
        emb_total += len(embedded)
        if not throughput_printed and payloads:
            rate = len(payloads) / max(time.time() - t0, 1e-9)
            print(f"embed throughput: {rate:.2f} clips/s "
                  f"({'packed' if PACKED is not None else 'reference'} path)", flush=True)
            throughput_printed = True
        flush()
        if bi % 10 == 0:
            print(f"batch {bi + 1}/{len(batches)} | embedded {emb_total:,} | "
                  f"perm-misses {len(set(new_misses))} | "
                  f"{(time.time() - T_SESSION) / 3600:.1f}h", flush=True)
        if (time.time() - T_SESSION) / 3600 > CFG["time_budget_h"]:
            print(f"time budget reached at batch {bi + 1} — stopping cleanly")
            break
flush(force=True)
print(f"shard {CFG['shard']} done: {emb_total:,} embedded this session")
print("files:", sorted(f.name for f in WORK.glob("ext_*")))
